# Batoid Pupil Comparison: Camera-Hexapod 8 mm vs Camera 4 mm + M2 4 mm

**Author:** Aaron Roodman
**Date Created:** 2026-06-15
**Last Modified:** 2026-06-15
**Status:** In Progress
**Keywords:** batoid, pupil, donut, hexapod, M2, defocus, ray trace, spot diagram, AOS

## Description

Use **Batoid** + the LSST optical model to ask whether the giant-donut **pupil** differs
when an 8 mm defocus is made by **moving the Camera hexapod 8 mm** vs **moving the Camera
and M2 hexapods 4 mm each** (M2 is powered, so it contributes differently).

Method (per the plan): trace a **dense grid of rays** (a spot diagram with many spots)
from a point source at a chosen field angle through the defocused system to the detector
plane; the 2-D **histogram of ray positions is the geometric pupil image** (the donut).
Compare the two hexapod configurations: donut images, their difference, and azimuthally
averaged radial profiles.

Hexapod pistons are applied as rigid z-shifts of the `LSSTCamera` group and `M2` via
`withGloballyShiftedOptic` (a hexapod piston = a global z translation), so no
`batoid_rubin` data download is needed. Model options: design `LSST_r.yaml`, or as-built
`Rubin_v3.12_r.yaml` / `Rubin_v3.14_r.yaml`.

**Output:** comparison figure(s); no files written by default.

**Based on:** the giant-donut study (`wfs_giant_donut_fit.ipynb`). The real giant donuts
are at e.g. R30_S21 (field radius ≈ 1.69°), seq 340 (intra_8mm) / 349 (extra_8mm).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-15 | Aaron Roodman | Initial version: batoid spot-diagram pupil/donut, camera-8mm vs camera-4mm+M2-4mm |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Build Configurations & Trace](#trace)
5. [Compare Pupils](#compare)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# LSST optical model:
#   "LSST_r.yaml"       — nominal design (~v3.3)
#   "Rubin_v3.12_r.yaml" — as-built v3.12   (default)
#   "Rubin_v3.14_r.yaml" — as-built v3.14
model_yaml = "Rubin_v3.12_r.yaml"
wavelength = 620e-9              # metres (r-band ~620 nm)

# Field angle of the source (deg). The real giant donut on R30_S21 is at field
# radius ≈ 1.69° (corner raft). theta is in batoid's optic field-angle frame.
theta_x_deg = 1.69
theta_y_deg = 0.0

# Defocus configurations to compare: (label, camera_hexapod_dz_mm, M2_hexapod_dz_mm).
# defocus_sign flips intra/extra (the data has intra_8mm and extra_8mm).
CONFIGS = [
    ("camera +8mm",      8.0, 0.0),
    ("camera +4 + M2 +4", 4.0, 4.0),
]
defocus_sign = +1.0              # +1 or -1 (extra vs intra-focal sense)

# Ray grid / histogram
n_pupil = 800                    # rays across the pupil grid (n_pupil^2 spots traced)
hist_npix = 256                  # output donut image size (pixels)
hist_half_mm = 4.0               # half-width of the donut image (mm); donut ~3.5 mm radius

cmap = "viridis"


<a id='setup'></a>
## Setup & Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import batoid

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()


<a id='functions'></a>
## Helper Functions

In [ ]:
def build_telescope(fiducial, camera_dz_mm, m2_dz_mm, sign=1.0):
    """Apply camera- and M2-hexapod z pistons (mm) to a fiducial batoid Optic.

    A hexapod piston is modelled as a rigid global z-shift of the optic group
    (`LSSTCamera` moves the lenses+filter+detector together; `M2` moves the
    secondary). Returns a new Optic.
    """
    tel = fiducial
    if camera_dz_mm:
        tel = tel.withGloballyShiftedOptic("LSSTCamera", [0, 0, sign * camera_dz_mm * 1e-3])
    if m2_dz_mm:
        tel = tel.withGloballyShiftedOptic("M2", [0, 0, sign * m2_dz_mm * 1e-3])
    return tel


def trace_donut(tel, theta_x_deg, theta_y_deg, wavelength, n_pupil,
                npix, half_mm, center=None):
    """Trace a dense pupil ray grid to the detector; histogram positions -> donut image.

    Returns dict(image, extent, center_mm, span_mm, n_rays). The 2-D histogram of the
    (un-vignetted) ray positions is the geometric pupil image at this defocus.
    """
    rays = batoid.RayVector.asGrid(
        optic=tel, wavelength=wavelength,
        theta_x=np.deg2rad(theta_x_deg), theta_y=np.deg2rad(theta_y_deg), nx=n_pupil)
    tel.trace(rays)
    ok = ~rays.vignetted & ~rays.failed
    x = rays.x[ok] * 1e3; y = rays.y[ok] * 1e3        # mm
    cx, cy = (float(x.mean()), float(y.mean())) if center is None else center
    rng = [[cx - half_mm, cx + half_mm], [cy - half_mm, cy + half_mm]]
    H, xe, ye = np.histogram2d(x, y, bins=npix, range=rng)
    return {"image": H.T, "extent": [xe[0]-cx, xe[-1]-cx, ye[0]-cy, ye[-1]-cy],
            "center_mm": (cx, cy), "span_mm": float(x.max() - x.min()),
            "n_rays": int(ok.sum())}


def azimuthal_profile(image, half_mm, nbin=120):
    """Azimuthally averaged radial profile of a (centered) donut image.

    Returns (r_mm, mean_value). Assumes the donut is centred in the image.
    """
    n = image.shape[0]
    yy, xx = np.mgrid[0:n, 0:n]
    c = (n - 1) / 2.0
    r = np.hypot(xx - c, yy - c) * (2 * half_mm / n)   # mm from centre
    edges = np.linspace(0, half_mm, nbin + 1)
    idx = np.digitize(r.ravel(), edges) - 1
    vals = image.ravel()
    rc, prof = [], []
    for k in range(nbin):
        sel = idx == k
        if sel.any():
            rc.append(0.5 * (edges[k] + edges[k + 1])); prof.append(vals[sel].mean())
    return np.array(rc), np.array(prof)


<a id='trace'></a>
## Build Configurations & Trace

In [ ]:
fiducial = batoid.Optic.fromYaml(model_yaml)
print("model:", model_yaml, "| field (deg):", (theta_x_deg, theta_y_deg),
      "| sign:", defocus_sign)

donuts = []
for label, cam_dz, m2_dz in CONFIGS:
    tel = build_telescope(fiducial, cam_dz, m2_dz, sign=defocus_sign)
    d = trace_donut(tel, theta_x_deg, theta_y_deg, wavelength, n_pupil,
                    hist_npix, hist_half_mm)
    donuts.append((label, d))
    print(f"  {label:>20}:  span {d['span_mm']:.2f} mm, {d['n_rays']} rays")


<a id='compare'></a>
## Compare Pupils

In [ ]:
# Donut images for each config + the difference of the first two.
n = len(donuts)
fig, axes = plt.subplots(1, n + 1, figsize=(4.6 * (n + 1), 4.6), constrained_layout=True)
vmax = max(d["image"].max() for _, d in donuts)
for ax, (label, d) in zip(axes, donuts):
    im = ax.imshow(d["image"], origin="lower", cmap=cmap, vmin=0, vmax=vmax,
                   extent=d["extent"])
    ax.set_title(f"{label}\nspan {d['span_mm']:.2f} mm", fontsize=10)
    ax.set_xlabel("mm"); ax.set_ylabel("mm")
    fig.colorbar(im, ax=ax, fraction=0.046)

(lA, dA), (lB, dB) = donuts[0], donuts[1]
diff = dA["image"] - dB["image"]
v = np.nanpercentile(np.abs(diff), 99.5)
im = axes[-1].imshow(diff, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v,
                     extent=dA["extent"])
axes[-1].set_title(f"({lA}) − ({lB})", fontsize=10)
axes[-1].set_xlabel("mm"); axes[-1].set_ylabel("mm")
fig.colorbar(im, ax=axes[-1], fraction=0.046)
fig.suptitle(f"Batoid pupil/donut — {model_yaml}, field ({theta_x_deg}, {theta_y_deg})°",
             fontsize=12)
plt.show()


In [ ]:
# Azimuthally averaged radial profiles — quantifies the size / illumination difference.
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for label, d in donuts:
    r, prof = azimuthal_profile(d["image"], hist_half_mm)
    ax.plot(r, prof, label=label)
ax.set_xlabel("radius from donut centre [mm]")
ax.set_ylabel("rays / pixel (illumination)")
ax.set_title("Azimuthally averaged pupil/donut profile")
ax.grid(alpha=0.3); ax.legend()
plt.show()

# Quick numbers: outer-edge radius (half-max on the falling outer edge) per config.
for label, d in donuts:
    r, prof = azimuthal_profile(d["image"], hist_half_mm)
    half = 0.5 * prof.max()
    outer = r[np.where(prof > half)[0].max()] if np.any(prof > half) else np.nan
    print(f"  {label:>20}: outer radius ~{outer:.2f} mm")
